**Purpose**: This notebook serves as the "chunking engine" for the Finance RAG project. It reads parsed JSON financial reports and a YAML configuration file to produce a set of structured, "analyst-centric" chunks. The entire process is logged as an MLflow experiment for versioning and reproducibility.

## 1. Imports
This cell imports all the libraries we need for the entire notebook. This ensures our environment is set up correctly from the start.

In [105]:
import json
import yaml
import uuid
import mlflow
from pathlib import Path
from typing import List, Dict, Any, Callable
import time
import logging

# Setup basic logging to see clear INFO or ERROR messages
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

## 2. Configuration Loading
This cell defines where our data lives and ensures the output folders are ready.

In [106]:
# --- Configuration Paths ---
STRATEGY_DIR = Path("../configs/strategies/")
INPUT_DIR = Path("../data/parsed/gen2/")
OUTPUT_DIR = Path("../data/chunks/")

# Ensure output directory exists
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Filter out the batch_report and get only individual company JSONs
source_files = [f for f in INPUT_DIR.glob("*.json") if "batch_report" not in f.name.lower()]

logging.info(f"Initialized Paths. Found {len(source_files)} company files to process.")

2026-04-21 11:13:29,136 - INFO - Initialized Paths. Found 2 company files to process.


## 3.The FinanceDataEngine Class
This class finds the correct "Periods" in the JSON and extracts the specific financial facts defined in the YAML.

In [107]:
def fetch_facts_for_chunk(data: Dict[str, Any], chunk_def: Dict[str, Any]) -> Dict[str, Any]:
    periods_by_id = {p['context_id']: p for p in data.get('periods', [])}

    raw_facts = {}
    
    # 1. Identify matching periods based on patterns
    patterns = chunk_def.get('context_pattern', [])
    if not isinstance(patterns, list): patterns = [patterns]
    
    target_periods = []
    for pat in patterns:
        if '*' in pat:
            prefix = pat.split('*')[0]
            target_periods.extend([p for cid, p in periods_by_id.items() if cid.startswith(prefix)])
        elif pat in periods_by_id:
            target_periods.append(periods_by_id[pat])

    # 2. Extract facts
    for period in target_periods:
        for fact in chunk_def.get('facts', []):
            val = period.get('numeric_facts', {}).get(fact, period.get('inf_facts', {}).get(fact))
            if val is not None: 
                raw_facts[fact] = val
    
    # 3. Handle extra_mappings if enabled
    if 'extra_mappings' in chunk_def:
        for mapping_name, rules in chunk_def['extra_mappings'].items():
            mapping_data = {}
            
            # Ensure rules is treated as a list even if only one is provided
            if not isinstance(rules, list):
                rules = [rules]
                
            for rule in rules:
                p = periods_by_id.get(rule.get('context_pattern'))
                if p:
                    # Look for the fact in numeric_facts or inf_facts
                    fact_key = rule.get('fact')
                    val = p.get('numeric_facts', {}).get(fact_key, p.get('inf_facts', {}).get(fact_key))
                    
                    if val is not None:
                        # Use the 'description' from YAML as the key in the output
                        mapping_data[rule.get('description', fact_key)] = val
                        
            raw_facts[mapping_name] = mapping_data
            
    return raw_facts

## 4. The ChunkSerializer Class
This class converts the raw data into clean text for the LLM and a structured payload for the database.

In [108]:
def serialize_chunk(chunk_name: str, chunk_def: Dict[str, Any], raw_facts: Dict[str, Any], file_name: str, metadata: Dict[str, Any]) -> Dict[str, Any]:
    # Generate ID
    chunk_id = f"{metadata['scrip_code']}_{file_name}_{chunk_name}"
    
    # Format Text Content
    text_content = f"{metadata['company_name']} - {chunk_def.get('description')}:\n"
    
    for fact, val in raw_facts.items():
        # 1. Check if this is a NESTED mapping (a dictionary of facts)
        if isinstance(val, dict) and 'value' not in val:
            header = fact.replace('_', ' ').title()
            text_content += f"  {header}:\n"
            
            # Loop through the nested items
            for sub_desc, sub_val in val.items():
                num = sub_val['value'] if isinstance(sub_val, dict) and 'value' in sub_val else sub_val
                text_content += f"    - {sub_desc}: {num:,.2f}\n"
        
        # 2. Handle standard facts
        else:
            num = val['value'] if isinstance(val, dict) and 'value' in val else val
            if isinstance(num, (int, float)):
                text_content += f"- {fact}: {num:,.2f}\n"
            else:
                text_content += f"- {fact}: {num}\n"

    return {
        "id": chunk_id,
        "text_content": text_content.strip(),
        "payload": {
            "metadata": {**metadata, "chunk_name": chunk_name, "source": file_name},
            "raw_data": raw_facts
        }
    }

## 5. MLflow Experiment Execution
This handles dynamic YAML loading per company and tracks everything in MLflow.

In [109]:
mlflow.set_experiment("Nifty50_Chunking_v2")

with mlflow.start_run() as run:
    start_time = time.time()
    all_processed_chunks = []
    
    for file_path in source_files:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            
        scrip = str(data.get('scrip_code'))
        strategy_file = STRATEGY_DIR / f"{scrip}.yaml"
        
        if not strategy_file.exists():
            logging.warning(f"Skipping {file_path.name}: No strategy for {scrip}.")
            continue
            
        with open(strategy_file, 'r', encoding='utf-8') as f:
            strategy = yaml.safe_load(f)
            
        report_meta = {
            "fiscal_year": data.get("fiscal_year"),
            "quarter": data.get("quarter"),
            "period_type": data.get("period_type")
        }
        combined_meta = {**strategy['company_metadata'], **report_meta}
        
        file_chunk_count = 0
        for name, definition in strategy['chunk_definitions'].items():
            facts = fetch_facts_for_chunk(data, definition)
            
            if facts:
                chunk = serialize_chunk(name, definition, facts, file_path.name, combined_meta)
                all_processed_chunks.append(chunk)
                file_chunk_count += 1
        
        logging.info(f"Processed {file_path.name}: {file_chunk_count} chunks created.")

    output_file = OUTPUT_DIR / "chunks_to_embed.jsonl"
    with open(output_file, 'w', encoding='utf-8') as f:
        for c in all_processed_chunks:
            f.write(json.dumps(c) + "\n")

    end_time = time.time()
    processing_time = end_time - start_time

    mlflow.log_metric("total_chunks_created", len(all_processed_chunks))
    mlflow.log_metric("processing_time_seconds", round(processing_time, 2))
    mlflow.log_artifact(str(output_file))
    logging.info(f"MLflow Run Complete: {len(all_processed_chunks)} total chunks created.")

2026-04-21 11:13:33,114 - INFO - Processed INTEGRATED_FILING_BANKING_1603568_17012026092624_WEB_parsed.json: 15 chunks created.
2026-04-21 11:13:33,242 - INFO - Processed RELIANCE_1425445_25042025095716_WEB_parsed.json: 21 chunks created.
2026-04-21 11:13:33,432 - INFO - MLflow Run Complete: 36 total chunks created.


## 6. Final Verification
A quick sanity check to see exactly what we've produced.

In [110]:
if  all_processed_chunks:
    
    print("--- Verification Passed ---")
    
    # 1. Take the first chunk from the in-memory list for inspection
    sample_chunk = all_processed_chunks[0]
    
    # 2. Print key details of the sample chunk to verify its structure
    print(f"\nSample Chunk ID: {sample_chunk['id']}")
    
    print("\nSample Text Content (first 250 chars):")
    print(f"{sample_chunk['text_content'][:250]}...")
    
    print("\nSample Payload Metadata Keys:")
    print(list(sample_chunk['payload']['metadata'].keys()))
    
    # 3. Final summary message
    print(f"\n Successfully created a total of {len(all_processed_chunks)} chunks.")
    print(f"   Output file saved to: {output_file}")
    if 'run' in locals():
        print(f"   MLflow Run ID: {run.info.run_id}")

else:
    print("--- Verification FAILED ---")
    print(" No chunks were generated. Please review the logs in the previous cells for errors.")
    print("   Common issues: YAML configuration mismatch or no source JSON files found.")

--- Verification Passed ---

Sample Chunk ID: 500180_INTEGRATED_FILING_BANKING_1603568_17012026092624_WEB_parsed.json_chunk_pnl_quarterly_interest_income

Sample Text Content (first 250 chars):
HDFC BANK LIMITED - Quarterly Interest Income (Core Banking Revenue):
- InterestEarned: 8,706,694.00
- InterestOrDiscountOnAdvancesOrBills: 6,411,401.00
- InterestOnBalancesWithReserveBankOfIndiaAndOtherInterBankFunds: 77,353.00
- OtherInterest: 159,...

Sample Payload Metadata Keys:
['company_name', 'scrip_code', 'industry', 'currency', 'report_type', 'fiscal_year_end', 'fiscal_year', 'quarter', 'period_type', 'chunk_name', 'source']

 Successfully created a total of 36 chunks.
   Output file saved to: ..\data\chunks\chunks_to_embed.jsonl
   MLflow Run ID: 8815e203b7d04e6b92974b435c7e3bb8
